# 01 — Feature Block Exploration

This notebook explores the structure and distribution of each feature block before training. Goals:
- Understand feature sparsity and dimensionality
- Identify batch effects or genome size confounders
- Visualise class separation with PCA/UMAP per block
- Check for block-level correlations with taxonomy vs. ecology

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project imports
from fungal_classifier.utils.io import load_metadata, load_feature_blocks

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})
sns.set_style("whitegrid")


## 1. Load metadata and feature matrices

In [ ]:
METADATA_PATH = Path("../data/raw/metadata.tsv")
FEATURES_DIR  = Path("../data/features/")

metadata = load_metadata(METADATA_PATH)
print(f"Genomes: {len(metadata)}")
print(f"Columns: {list(metadata.columns)}")
metadata.head()


In [ ]:
blocks = load_feature_blocks(FEATURES_DIR)
print("Feature blocks loaded:")
for name, df in blocks.items():
    n_nonzero = (df > 0).sum(axis=1).mean()
    sparsity  = 1 - (df != 0).mean().mean()
    print(f"  {name:20s}  shape={str(df.shape):18s}  mean_nonzero/genome={n_nonzero:.1f}  sparsity={sparsity:.2%}")


## 2. Feature sparsity heatmaps

Visualize which features are present across genomes for each block.

In [ ]:
fig, axes = plt.subplots(1, len(blocks), figsize=(5 * len(blocks), 4))
if len(blocks) == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, blocks.items()):
    # Downsample columns for readability
    sub = (df > 0).sample(n=min(200, df.shape[1]), axis=1, random_state=42)
    # Sort rows by taxonomy
    order_col = "taxonomy_order"
    if order_col in metadata.columns:
        order = metadata.loc[sub.index.intersection(metadata.index), order_col].sort_values().index
        sub = sub.reindex(order.intersection(sub.index))
    sns.heatmap(sub.values.T, ax=ax, cmap="Blues", xticklabels=False, yticklabels=False,
                cbar_kws={"shrink": 0.5})
    ax.set_title(f"{name}\n({df.shape[1]} features)")
    ax.set_xlabel("Genomes")
    ax.set_ylabel("Features (sample)")

plt.suptitle("Feature Presence/Absence per Block", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


## 3. PCA per feature block coloured by taxonomy

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

TARGET = "taxonomy_order"  # change to "ecological_niche" etc.

fig, axes = plt.subplots(1, len(blocks), figsize=(5 * len(blocks), 4))
if len(blocks) == 1:
    axes = [axes]

palette = sns.color_palette("tab20", n_colors=metadata[TARGET].nunique())
color_map = dict(zip(sorted(metadata[TARGET].dropna().unique()), palette))

for ax, (name, df) in zip(axes, blocks.items()):
    common = df.index.intersection(metadata.index)
    X = StandardScaler().fit_transform(df.loc[common].fillna(0))
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X)
    labels = metadata.loc[common, TARGET].fillna("Unknown")
    colors = [color_map.get(l, "grey") for l in labels]
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=8, alpha=0.6)
    ax.set_title(f"{name}\nPC1 {pca.explained_variance_ratio_[0]:.1%} | PC2 {pca.explained_variance_ratio_[1]:.1%}")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

# Legend (shared)
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=7, label=l)
           for l, c in color_map.items()]
axes[-1].legend(handles=handles, bbox_to_anchor=(1.05, 1), loc="upper left",
                fontsize=7, title=TARGET)
plt.suptitle(f"PCA by Block — coloured by {TARGET}", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


## 4. UMAP per block (optional — slower)

In [ ]:
try:
    from umap import UMAP

    TARGET = "ecological_niche"
    fig, axes = plt.subplots(1, len(blocks), figsize=(5 * len(blocks), 4))
    if len(blocks) == 1:
        axes = [axes]

    eco_palette = sns.color_palette("Set2", n_colors=metadata[TARGET].nunique())
    eco_map = dict(zip(sorted(metadata[TARGET].dropna().unique()), eco_palette))

    for ax, (name, df) in zip(axes, blocks.items()):
        common = df.index.intersection(metadata.index)
        X = StandardScaler().fit_transform(df.loc[common].fillna(0))
        reducer = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
        coords = reducer.fit_transform(X)
        labels = metadata.loc[common, TARGET].fillna("Unknown")
        colors = [eco_map.get(l, "grey") for l in labels]
        ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=8, alpha=0.6)
        ax.set_title(f"{name}\nUMAP")
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")

    handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c,
                           markersize=7, label=l) for l, c in eco_map.items()]
    axes[-1].legend(handles=handles, bbox_to_anchor=(1.05, 1), fontsize=7,
                    title=TARGET, loc="upper left")
    plt.suptitle(f"UMAP by Block — coloured by {TARGET}", y=1.02)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("umap-learn not installed. Run: pip install umap-learn")


## 5. Between-block correlation

How correlated are the feature blocks? High correlation = redundancy.

In [ ]:
from sklearn.decomposition import PCA

# Reduce each block to 10 PCs, then compute correlation between PC1s
block_pc1 = {}
for name, df in blocks.items():
    common = df.index.intersection(metadata.index)
    X = StandardScaler().fit_transform(df.loc[common].fillna(0))
    pc1 = PCA(n_components=1, random_state=42).fit_transform(X)[:, 0]
    block_pc1[name] = pd.Series(pc1, index=common)

pc1_df = pd.DataFrame(block_pc1).dropna()
corr = pc1_df.corr()

fig, ax = plt.subplots(figsize=(6, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5)
ax.set_title("Block PC1 Pearson Correlations")
plt.tight_layout()
plt.show()


## 6. Class balance check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col in zip(axes, ["taxonomy_order", "ecological_niche"]):
    if col not in metadata.columns:
        continue
    counts = metadata[col].value_counts()
    counts.plot(kind="bar", ax=ax, color="#1a6fa8", edgecolor="white")
    ax.set_title(f"Class distribution: {col}")
    ax.set_xlabel("")
    ax.set_ylabel("# Genomes")
    ax.tick_params(axis="x", labelsize=8, rotation=45)
    # Annotate imbalance ratio
    ratio = counts.max() / counts.min()
    ax.text(0.98, 0.95, f"Max/min ratio: {ratio:.1f}x",
            transform=ax.transAxes, ha="right", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()


## 7. Genome size distribution (repeat block proxy)

In [ ]:
if "repeats" in blocks:
    rpt = blocks["repeats"]
    total_rpt = rpt[[c for c in rpt.columns if c.startswith("repeat_class_")]].sum(axis=1)
    meta_common = metadata.loc[total_rpt.index.intersection(metadata.index)]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(total_rpt, bins=50, color="#1a6fa8", edgecolor="white")
    axes[0].set_xlabel("Total repeat fraction")
    axes[0].set_ylabel("# Genomes")
    axes[0].set_title("Repeat content distribution")

    if "taxonomy_order" in meta_common.columns:
        top_orders = meta_common["taxonomy_order"].value_counts().head(8).index
        sub = total_rpt[meta_common.index[meta_common["taxonomy_order"].isin(top_orders)]]
        order_labels = meta_common.loc[sub.index, "taxonomy_order"]
        for order in top_orders:
            vals = sub[order_labels == order]
            axes[1].hist(vals, bins=20, alpha=0.5, label=order)
        axes[1].set_xlabel("Total repeat fraction")
        axes[1].set_title("Repeat content by order")
        axes[1].legend(fontsize=7)

    plt.tight_layout()
    plt.show()
else:
    print("Repeat block not loaded.")
